In [1]:
using Distributed

# Add workers (adjust to your CPU count)
addprocs(2)

@everywhere begin
    using SpeedyWeather
end

In [2]:
@everywhere function build_spectral_grid()
    spectral_grid = SpectralGrid()
    return spectral_grid
end

FOR SOLAR FLUX

In [ ]:
# @everywhere function run_single_flux(idx, solar_const, outdir)

#     spectral_grid = build_spectral_grid()

#     fname = joinpath(outdir, "output$(idx).nc")
#     output = NetCDFOutput(spectral_grid; filename=fname)

#     add!(output, SpeedyWeather.DivergenceOutput())
#     add!(output, SpeedyWeather.VorticityOutput())
#     add!(output, SpeedyWeather.TemperatureOutput())
#     add!(output, SpeedyWeather.SurfaceShortwaveDownOutput())
#     add!(output, SpeedyWeather.SoilMoistureOutput())
#     add!(output, SpeedyWeather.SoilTemperatureOutput())
#     add!(output, SpeedyWeather.SurfaceTemperatureOutput())
#     add!(output, SpeedyWeather.LandSeaMaskOutput())

#     p = Earth(spectral_grid, solar_constant=solar_const)
#     drag = SpeedLimitDrag(spectral_grid)
#     model = PrimitiveWetModel(spectral_grid; output=output, planet=p, drag=drag)

#     add!(model, SpeedyWeather.SnowDepthOutput())
#     add!(model, SpeedyWeather.SnowMeltOutput())
#     add!(model, SpeedyWeather.RadiationOutput())
#     add!(model, SpeedyWeather.OceanOutput())
#     add!(model, SpeedyWeather.PrecipitationOutput())
#     add!(model, SpeedyWeather.HumidityOutput())

#     sim = initialize!(model)
#     run!(sim, period=Year(2), output=true)

#     return fname
# end


FOR ATMOSPHERIC PRESSURE

In [6]:
@everywhere function run_single(idx, pressure, outdir)

    spectral_grid = build_spectral_grid()
    run_dir = joinpath(outdir, "run_000$(idx)")
    mkpath(run_dir)
    fname = joinpath(run_dir, "output.nc")
    output = NetCDFOutput(spectral_grid; filename=fname)

    add!(output, SpeedyWeather.DivergenceOutput())
    add!(output, SpeedyWeather.VorticityOutput())
    add!(output, SpeedyWeather.TemperatureOutput())
    add!(output, SpeedyWeather.SurfaceShortwaveDownOutput())
    add!(output, SpeedyWeather.SoilMoistureOutput())
    add!(output, SpeedyWeather.SoilTemperatureOutput())
    add!(output, SpeedyWeather.SurfaceTemperatureOutput())
    add!(output, SpeedyWeather.LandSeaMaskOutput())

    atmos = EarthAtmosphere(spectral_grid, pressure_reference = pressure)
    drag = SpeedLimitDrag(spectral_grid)
    model = PrimitiveWetModel(spectral_grid; output=output, atmosphere=atmos, drag=drag)

    add!(model, SpeedyWeather.SnowDepthOutput())
    add!(model, SpeedyWeather.SnowMeltOutput())
    add!(model, SpeedyWeather.RadiationOutput())
    add!(model, SpeedyWeather.OceanOutput())
    add!(model, SpeedyWeather.PrecipitationOutput())
    add!(model, SpeedyWeather.HumidityOutput())

    sim = initialize!(model)
    run!(sim, period=Year(2), output=true)

    return run_dir
end

In [7]:
# sc = 1365
# flux = sc .* [0.8,0.9,1.0]   # add more values if you want multiple runs
# pairs = collect(enumerate(flux))
outdir = "/Users/woodh/Documents/rotation_project"
p_ref = 100000.0
pressure = p_ref .* [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
pairs = collect(enumerate(pressure))

results = pmap(p -> run_single(p[1], p[2], outdir), pairs)

println(results)

      From worker 2:	┌ Warning: NaN or Inf detected at time step 1637
      From worker 2:	└ @ SpeedyWeather ~/.julia/packages/SpeedyWeather/hKf2u/src/output/feedback.jl:119
["/Users/woodh/Documents/rotation_project/run_0001", "/Users/woodh/Documents/rotation_project/run_0002", "/Users/woodh/Documents/rotation_project/run_0003", "/Users/woodh/Documents/rotation_project/run_0004", "/Users/woodh/Documents/rotation_project/run_0005", "/Users/woodh/Documents/rotation_project/run_0006", "/Users/woodh/Documents/rotation_project/run_0007", "/Users/woodh/Documents/rotation_project/run_0008"]
